# Experiment 3 - status overview of the 6 folds (read-only; a CPU runtime is enough)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import csv, os, time, re
import pandas as pd
ROOT = '/content/drive/MyDrive/Flood2/models/exp3'; TOTAL = 46
rows = []
for f in ['irma','matthew','ian','milton','dorian','beryl']:
    d = f'{ROOT}/fold_{f}'; mp = f'{d}/manifest.csv'; lg = f'{d}/train_stdout.log'; r = {'fold': f}
    man = list(csv.DictReader(open(mp))) if os.path.exists(mp) else []
    done = [x for x in man if x['status'] == 'done']; r['completed configs'] = f'{len(done)} / {TOTAL}'
    if os.path.exists(lg):
        age = time.time() - os.path.getmtime(lg); txt = open(lg, errors='replace').read()
        r['status'] = 'all done' if 'fold complete' in txt else ('running' if age < 300 else 'interrupted, needs resume')
        m = re.findall(r'progress (\d+)/(\d+) elapsed ([\d:]+) ETA ([\d:]+)', txt)
        if m: r['session progress'] = f'{m[-1][0]}/{m[-1][1]}'; r['elapsed'] = m[-1][2]; r['est. remaining'] = m[-1][3]
        r['log updated'] = f'{age/60:.0f} min ago'
    else: r['status'] = 'not started'
    if done:
        secs = [float(x['train_sec']) + float(x['eval_sec']) for x in done]; r['mean time per config s'] = round(sum(secs) / len(secs))
    rows.append(r)
display(pd.DataFrame(rows).fillna(''))